In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Dropout

In [8]:
# CSV dosyasını okuyoruz
df = pd.read_csv('Tesla_Nasdaq_Prediction.csv')

# Tarihler yeniden eskiye olduğu için veriyi tersine çeviriyoruz!
df = df.iloc[::-1].reset_index(drop=True)

# Sadece Kapanış (Close/Last) fiyatlarını alıp matrise çeviriyoruz
data = df['Close/Last'].values.reshape(-1, 1)

In [9]:
train_length = round(len(data) * 0.7)

train_data = data[:train_length]
val_data = data[train_length:]

In [10]:
scaler = MinMaxScaler(feature_range=(0,1))

# Eğitim verisine hem fit hem transform yapıyoruz
scaled_train_data = scaler.fit_transform(train_data)

# Test verisine transform yapıyoruz
scaled_val_data = scaler.transform(val_data)

In [11]:
step = 50

def create_dataset(dataset, step):
    X, Y = [], []
    for i in range(step, len(dataset)):
        X.append(dataset[i-step:i, 0])
        Y.append(dataset[i, 0])
    return np.array(X), np.array(Y)

X_train, y_train = create_dataset(scaled_train_data, step)
X_val, y_val = create_dataset(scaled_val_data, step)

# Keras RNN 3 Boyutlu Veri Bekler: (Örnek Sayısı, Zaman Adımı, Özellik Sayısı)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))

In [12]:
model = Sequential()

# 1. Katman
model.add(SimpleRNN(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))

# 2. Katman
model.add(SimpleRNN(units=50, return_sequences=True))
model.add(Dropout(0.2))

# 3. Katman
model.add(SimpleRNN(units=50, return_sequences=True))
model.add(Dropout(0.2))

# 4. Katman (Son RNN)
model.add(SimpleRNN(units=50))
model.add(Dropout(0.2))

# Çıkış Katmanı
model.add(Dense(units=1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_val, y_val))

# Test verisi üzerinden tahmin yap
predicted_prices = model.predict(X_val)

# Tahminleri 0-1 aralığından çıkarıp gerçek Dolar seviyesine getir
predicted_prices = scaler.inverse_transform(predicted_prices)
real_prices = scaler.inverse_transform(y_val.reshape(-1, 1))

Epoch 1/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 13s 67ms/step - loss: 0.3507 - val_loss: 66.7815
Epoch 2/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 0.1637 - val_loss: 64.5566
Epoch 3/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 0.1009 - val_loss: 62.9678
Epoch 4/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - loss: 0.0698 - val_loss: 62.0871
Epoch 5/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 0.0519 - val_loss: 60.3430
Epoch 6/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - loss: 0.0433 - val_loss: 60.3723
Epoch 7/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 9s 56ms/step - loss: 0.0316 - val_loss: 59.0047
Epoch 8/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 0.0269 - val_loss: 60.4332
Epoch 9/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0209 - val_loss: 58.9125
Epoch 10/10
107/107 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - loss: 0.0169 - val_loss: 57.6238
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
